# UdaciSense: Optimized Object Recognition

## Notebook 3: Multi-step Optimization Pipeline

**🎯 Sequential Distillation → Quantization Pipeline**

This notebook implements a multi-stage optimization pipeline combining:
1. **Knowledge Distillation**: Teacher-student training for architecture compression
2. **Dynamic Quantization**: INT8 precision reduction for mobile deployment

**CTO Requirements:**
- Model should be **70% smaller** than baseline (5.96 MB → 1.79 MB)
- Model should **reduce inference time by 60%** (10.56 ms → 4.22 ms)
- Model should **maintain accuracy within 5%** of baseline (≥83% from 87.40%)

**Pipeline Strategy**: Sequential application optimizes each technique's effectiveness while maintaining accuracy.

### Step 1: Set up the environment

In [ ]:
# Mount Google Drive and Setup
from google.colab import drive
import os
import sys
import warnings
import random
import numpy as np
import torch
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# UPDATE THIS PATH to your Google Drive project location
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/udacity-ml-compression-pipeline/project/starter_kit'
os.chdir(DRIVE_PROJECT_PATH)
print(f"✅ Changed to directory: {os.getcwd()}")

# Add to Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
src_dir = os.path.join(current_dir, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

# Set deterministic mode for reproducibility
def set_deterministic_mode(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_deterministic_mode(42)

In [ ]:
# Install required packages with UV (faster installation)
!curl -LsSf https://astral.sh/uv/install.sh | sh
!/root/.local/bin/uv pip install --system torch>=2.0.0 torchvision>=0.15.0
!/root/.local/bin/uv pip install --system matplotlib seaborn pandas scikit-learn pillow tqdm plotly thop

print("✅ All packages installed successfully!")

In [ ]:
# Device setup and detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cpu_device = torch.device('cpu')

# Check available devices
devices = ["cpu"]
if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    devices.extend([f"cuda:{i} ({torch.cuda.get_device_name(i)})" for i in range(num_devices)])
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🚀 GPU Available: {torch.cuda.get_device_name(0)}")
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
    torch.cuda.empty_cache()
else:
    print("⚠️ No GPU found, using CPU")

print(f"Devices available: {devices}")
print(f"Primary device: {device}")

### Step 2: Import modules and load dataset

In [ ]:
# Import project modules
import json
import matplotlib.pyplot as plt
import pandas as pd
import time
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Import project-specific modules
from src.utils import MAX_ALLOWED_ACCURACY_DROP, TARGET_INFERENCE_SPEEDUP, TARGET_MODEL_COMPRESSION
from src.utils.data_loader import get_household_loaders, print_dataloader_stats, visualize_batch
from src.utils.model import load_model, save_model, print_model_summary
from src.utils.compression import evaluate_optimized_model, compare_optimized_model_to_baseline
from src.utils.evaluation import evaluate_model_metrics
from src.compression.in_training.distillation import train_with_distillation, MobileNetV3_Household_Small

print("✅ All modules imported successfully")
print(f"🎯 Targets: {TARGET_MODEL_COMPRESSION*100}% size reduction, {TARGET_INFERENCE_SPEEDUP*100}% speedup, <{MAX_ALLOWED_ACCURACY_DROP*100}% accuracy drop")

In [ ]:
# Load household objects dataset
train_loader, test_loader = get_household_loaders(
    image_size="CIFAR", 
    batch_size=256, 
    num_workers=2
)
class_names = train_loader.dataset.classes
input_size = (1, 3, 32, 32)

print(f"✅ Dataset loaded: {len(class_names)} classes")
print(f"Classes: {class_names}")
print(f"Input size: {input_size}")

# Display dataset statistics
for dataset_type, data_loader in [('train', train_loader), ('test', test_loader)]:
    print(f"\n{dataset_type.title()} set information:")
    print_dataloader_stats(data_loader, dataset_type)

# Visualize sample images
print("\nSample images from training set:")
visualize_batch(train_loader, num_images=8)

### Step 3: Load baseline model and establish targets

In [ ]:
# Load baseline model and metrics
print("📊 Loading baseline model and metrics...")

baseline_model_path = "models/baseline_mobilenet_colab/checkpoints/model.pth"
baseline_metrics_path = "results/baseline_mobilenet_colab/metrics.json"

baseline_model = load_model(baseline_model_path, device)
with open(baseline_metrics_path, 'r') as f:
    baseline_metrics = json.load(f)

print_model_summary(baseline_model)

# Calculate optimization targets
target_size_mb = baseline_metrics['size']['model_size_mb'] * (1 - TARGET_MODEL_COMPRESSION)
target_cpu_time = baseline_metrics['timing']['cpu']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
min_accuracy = baseline_metrics['accuracy']['top1_acc'] * (1 - MAX_ALLOWED_ACCURACY_DROP)

print(f"\n{'='*60}")
print("BASELINE PERFORMANCE & CTO TARGETS")
print(f"{'='*60}")
print(f"📋 BASELINE METRICS:")
print(f"   Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"   Size: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"   CPU Time: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms")

print(f"\n🎯 CTO OPTIMIZATION TARGETS:")
print(f"   1. Size: {baseline_metrics['size']['model_size_mb']:.2f} → {target_size_mb:.2f} MB ({TARGET_MODEL_COMPRESSION*100:.0f}% reduction)")
print(f"   2. Speed: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} → {target_cpu_time:.2f} ms ({TARGET_INFERENCE_SPEEDUP*100:.0f}% faster)")
print(f"   3. Accuracy: ≥ {min_accuracy:.2f}% (within {MAX_ALLOWED_ACCURACY_DROP*100:.0f}% of baseline)")
print(f"{'='*60}")

### Step 4: Implement multi-stage optimization pipeline

Based on analysis from previous notebooks, we implement a sequential pipeline combining:
1. **Knowledge Distillation**: Compress architecture while preserving knowledge
2. **Dynamic Quantization**: Reduce precision for mobile deployment

This approach maximizes compression while maintaining accuracy through careful sequencing.

In [ ]:
# Multi-stage optimization pipeline implementation
class OptimizedCompressionPipeline:
    """Sequential optimization pipeline combining template structure with proven implementations"""
    
    def __init__(self, name, baseline_model, baseline_metrics, train_loader, test_loader, class_names, input_size, device):
        self.name = name
        self.baseline_model = baseline_model
        self.baseline_metrics = baseline_metrics
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.class_names = class_names
        self.input_size = input_size
        self.device = device
        self.cpu_device = torch.device('cpu')
        
        self.results_history = []
        self.models = {}
        
        # Create directories
        self.model_dir = f"models/pipeline/{name}"
        self.results_dir = f"results/pipeline/{name}"
        for d in [self.model_dir, self.results_dir]:
            os.makedirs(d, exist_ok=True)
    
    def stage1_knowledge_distillation(self):
        """Stage 1: Knowledge Distillation with smaller student model"""
        print("\n🔄 STAGE 1: Knowledge Distillation")
        print("=" * 50)
        
        # Create student model
        student_model = MobileNetV3_Household_Small(num_classes=len(self.class_names))
        student_model = student_model.to(self.device)
        
        # Training configuration with actual objects
        optimizer = optim.Adam(student_model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        
        training_config = {
            'temperature': 4.0,
            'alpha': 0.7,
            'num_epochs': 20,
            'optimizer': optimizer,
            'criterion': criterion,
            'scheduler': scheduler,
            'patience': 10,
            'grad_clip_norm': 1.0,
            'device': self.device
        }
        
        # Train with distillation
        checkpoint_path = f"{self.model_dir}/stage1_distilled_model.pth"
        student_model, training_stats, best_accuracy, best_epoch = train_with_distillation(
            student_model=student_model,
            teacher_model=self.baseline_model,
            train_loader=self.train_loader,
            test_loader=self.test_loader,
            training_config=training_config,
            checkpoint_path=checkpoint_path
        )
        
        # Evaluate stage 1
        stage1_results = self.evaluate_stage(student_model, "stage1_distillation")
        self.results_history.append(('Stage 1: Distillation', stage1_results))
        self.models['distilled'] = student_model
        
        return student_model, stage1_results
    
    def stage2_quantization(self, input_model):
        """Stage 2: Dynamic Quantization"""
        print("\n🔄 STAGE 2: Dynamic Quantization")
        print("=" * 50)
        
        # Move to CPU for quantization
        model_cpu = input_model.to(self.cpu_device)
        model_cpu.eval()
        
        # Apply dynamic quantization
        quantized_model = torch.quantization.quantize_dynamic(
            model_cpu,
            {torch.nn.Linear, torch.nn.Conv2d},
            dtype=torch.qint8
        )
        
        # Save quantized model
        save_path = f"{self.model_dir}/final_quantized_model.pth"
        torch.save(quantized_model.state_dict(), save_path)
        print(f"💾 Saved quantized model to {save_path}")
        
        # Evaluate stage 2
        stage2_results = self.evaluate_stage(quantized_model, "stage2_quantization", device_eval=self.cpu_device)
        self.results_history.append(('Stage 2: Quantization', stage2_results))
        self.models['final'] = quantized_model
        
        return quantized_model, stage2_results
    
    def evaluate_stage(self, model, stage_name, device_eval=None):
        """Evaluate pipeline stage with comprehensive metrics"""
        if device_eval is None:
            device_eval = self.device
            
        print(f"📊 Evaluating {stage_name}...")
        
        experiment_name = f"pipeline/{stage_name}"
        results = evaluate_optimized_model(
            model, 
            self.test_loader, 
            experiment_name, 
            self.class_names, 
            self.input_size,
            device=device_eval
        )
        
        # Check requirements
        self.check_requirements(results, stage_name)
        
        return results
    
    def check_requirements(self, current_metrics, stage_name):
        """Check if current metrics meet CTO requirements"""
        print(f"\n🎯 {stage_name} vs CTO Requirements:")
        
        baseline_size = self.baseline_metrics['size']['model_size_mb']
        baseline_acc = self.baseline_metrics['accuracy']['top1_acc']
        baseline_time = self.baseline_metrics['timing']['cpu']['avg_time_ms']
        
        # Size requirement
        size_reduction = (1 - current_metrics['size']['model_size_mb'] / baseline_size) * 100
        size_meets = size_reduction >= TARGET_MODEL_COMPRESSION * 100
        print(f"  Size reduction: {size_reduction:.1f}% (target: {TARGET_MODEL_COMPRESSION*100:.0f}%) {'✅' if size_meets else '❌'}")
        
        # Speed requirement  
        speed_improvement = (1 - current_metrics['timing']['cpu']['avg_time_ms'] / baseline_time) * 100
        speed_meets = speed_improvement >= TARGET_INFERENCE_SPEEDUP * 100
        print(f"  Speed improvement: {speed_improvement:.1f}% (target: {TARGET_INFERENCE_SPEEDUP*100:.0f}%) {'✅' if speed_meets else '❌'}")
        
        # Accuracy requirement
        accuracy_change = current_metrics['accuracy']['top1_acc'] - baseline_acc
        accuracy_meets = accuracy_change >= -MAX_ALLOWED_ACCURACY_DROP * baseline_acc
        print(f"  Accuracy change: {accuracy_change:+.1f}pp (max drop: {MAX_ALLOWED_ACCURACY_DROP*baseline_acc:.1f}pp) {'✅' if accuracy_meets else '❌'}")
        
        return size_meets and speed_meets and accuracy_meets
    
    def run_pipeline(self):
        """Execute complete optimization pipeline"""
        print(f"\n{'='*70}")
        print(f"🚀 RUNNING PIPELINE: {self.name}")
        print(f"{'='*70}")
        
        # Stage 1: Knowledge Distillation
        distilled_model, stage1_results = self.stage1_knowledge_distillation()
        
        # Stage 2: Quantization
        final_model, stage2_results = self.stage2_quantization(distilled_model)
        
        # Generate summary
        self.generate_pipeline_summary()
        
        return final_model, self.results_history
    
    def generate_pipeline_summary(self):
        """Generate comprehensive pipeline results summary"""
        print("\n" + "=" * 70)
        print("📊 PIPELINE RESULTS SUMMARY")
        print("=" * 70)
        
        baseline_size = self.baseline_metrics['size']['model_size_mb']
        baseline_acc = self.baseline_metrics['accuracy']['top1_acc']
        baseline_time = self.baseline_metrics['timing']['cpu']['avg_time_ms']
        
        print(f"📋 BASELINE: {baseline_acc:.2f}% acc, {baseline_size:.2f} MB, {baseline_time:.2f} ms")
        
        for stage_name, results in self.results_history:
            # Handle both tuple and dict formats
            if isinstance(results, tuple):
                results = results[0] if len(results) > 0 else results
            
            acc = results['accuracy']['top1_acc']
            size = results['size']['model_size_mb']
            cpu_time = results['timing']['cpu']['avg_time_ms']
            
            size_reduction = (1 - size/baseline_size) * 100
            acc_drop = baseline_acc - acc
            speed_improvement = (1 - cpu_time/baseline_time) * 100
            
            print(f"📋 {stage_name}: {acc:.2f}% acc ({acc_drop:+.2f}%), {size:.2f} MB ({size_reduction:.1f}% ↓), {cpu_time:.2f} ms ({speed_improvement:+.1f}%)")
        
        # Final assessment
        if self.results_history:
            final_results = self.results_history[-1][1]
            # Handle tuple format
            if isinstance(final_results, tuple):
                final_results = final_results[0] if len(final_results) > 0 else final_results
            
            final_meets_all = self.check_final_requirements(final_results)
            
            print(f"\n🏆 FINAL RESULT: {'✅ ALL REQUIREMENTS MET' if final_meets_all else '⚠️ PARTIAL SUCCESS'}")
    
    def check_final_requirements(self, final_results):
        """Check if final results meet all CTO requirements"""
        baseline_size = self.baseline_metrics['size']['model_size_mb']
        baseline_acc = self.baseline_metrics['accuracy']['top1_acc']
        baseline_time = self.baseline_metrics['timing']['cpu']['avg_time_ms']
        
        target_size = baseline_size * (1 - TARGET_MODEL_COMPRESSION)
        target_acc = baseline_acc * (1 - MAX_ALLOWED_ACCURACY_DROP)
        target_time = baseline_time * (1 - TARGET_INFERENCE_SPEEDUP)
        
        size_meets = final_results['size']['model_size_mb'] <= target_size
        acc_meets = final_results['accuracy']['top1_acc'] >= target_acc
        speed_meets = final_results['timing']['cpu']['avg_time_ms'] <= target_time
        
        return size_meets and acc_meets and speed_meets

print("✅ Optimized pipeline class defined")

### Step 5: Execute optimization pipeline

In [ ]:
# Execute complete optimization pipeline
print("🚀 Initializing and running complete optimization pipeline...")

# Initialize pipeline with improved class
pipeline = OptimizedCompressionPipeline(
    name="distillation_quantization",
    baseline_model=baseline_model,
    baseline_metrics=baseline_metrics,
    train_loader=train_loader,
    test_loader=test_loader,
    class_names=class_names,
    input_size=input_size,
    device=device
)

# Run complete pipeline
final_optimized_model, pipeline_results = pipeline.run_pipeline()

print("\n🎉 Pipeline execution completed successfully!")

### Step 6: Analyze results and visualize pipeline performance

In [ ]:
# Generate comprehensive analysis and visualizations
print("📊 Generating final analysis and visualizations...")

# Create comparison DataFrame
comparison_data = []

# Add baseline
comparison_data.append({
    'Model': 'Baseline',
    'Accuracy (%)': baseline_metrics['accuracy']['top1_acc'],
    'Size (MB)': baseline_metrics['size']['model_size_mb'],
    'CPU Time (ms)': baseline_metrics['timing']['cpu']['avg_time_ms'],
    'Size Reduction (%)': 0.0,
    'Accuracy Drop (%)': 0.0
})

# Add pipeline stages
baseline_size = baseline_metrics['size']['model_size_mb']
baseline_acc = baseline_metrics['accuracy']['top1_acc']

for stage_name, results in pipeline_results:
    size_reduction = (1 - results['size']['model_size_mb']/baseline_size) * 100
    acc_drop = baseline_acc - results['accuracy']['top1_acc']
    
    comparison_data.append({
        'Model': stage_name,
        'Accuracy (%)': results['accuracy']['top1_acc'],
        'Size (MB)': results['size']['model_size_mb'],
        'CPU Time (ms)': results['timing']['cpu']['avg_time_ms'],
        'Size Reduction (%)': size_reduction,
        'Accuracy Drop (%)': acc_drop
    })

# Create and display comparison table
df_comparison = pd.DataFrame(comparison_data)
print("\n📊 COMPLETE PIPELINE COMPARISON:")
print(df_comparison.round(2))

# Create visualization
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

models = df_comparison['Model']
colors = ['blue', 'orange', 'green']

# Plot 1: Model Size
bars1 = ax1.bar(models, df_comparison['Size (MB)'], color=colors, alpha=0.7)
ax1.set_title('Model Size Progression', fontsize=14, fontweight='bold')
ax1.set_ylabel('Size (MB)')
ax1.axhline(y=target_size_mb, color='red', linestyle='--', label=f'Target: {target_size_mb:.1f} MB')
ax1.legend()
for i, (bar, size) in enumerate(zip(bars1, df_comparison['Size (MB)'])):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
             f'{size:.2f}', ha='center', va='bottom', fontweight='bold')

# Plot 2: Inference Time  
bars2 = ax2.bar(models, df_comparison['CPU Time (ms)'], color=colors, alpha=0.7)
ax2.set_title('Inference Time Progression', fontsize=14, fontweight='bold')
ax2.set_ylabel('Inference Time (ms)')
ax2.axhline(y=target_cpu_time, color='red', linestyle='--', 
           label=f'Target: {target_cpu_time:.1f} ms')
ax2.legend()
for i, (bar, time) in enumerate(zip(bars2, df_comparison['CPU Time (ms)'])):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{time:.1f}', ha='center', va='bottom', fontweight='bold')

# Plot 3: Accuracy
bars3 = ax3.bar(models, df_comparison['Accuracy (%)'], color=colors, alpha=0.7)
ax3.set_title('Accuracy Progression', fontsize=14, fontweight='bold') 
ax3.set_ylabel('Top-1 Accuracy (%)')
ax3.axhline(y=min_accuracy, color='red', linestyle='--', 
           label=f'Min Acceptable: {min_accuracy:.1f}%')
ax3.legend()
for i, (bar, acc) in enumerate(zip(bars3, df_comparison['Accuracy (%)'])):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, 
             f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

# Plot 4: Size Reduction
bars4 = ax4.bar(models, df_comparison['Size Reduction (%)'], color=colors, alpha=0.7)
ax4.set_title('Cumulative Size Reduction', fontsize=14, fontweight='bold')
ax4.set_ylabel('Size Reduction (%)')
ax4.axhline(y=TARGET_MODEL_COMPRESSION*100, color='red', linestyle='--', 
           label=f'Target: {TARGET_MODEL_COMPRESSION*100:.0f}%')
ax4.legend()
for i, (bar, reduction) in enumerate(zip(bars4, df_comparison['Size Reduction (%)'])):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{reduction:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('results/pipeline_visualization.png', dpi=300, bbox_inches='tight')
plt.show()

# Save results
df_comparison.to_csv('results/pipeline_comparison.csv', index=False)
print("\n💾 Results saved to results/pipeline_comparison.csv")

print("\n🎉 PIPELINE ANALYSIS COMPLETE!")

## Multi-Stage Optimization Analysis

### Pipeline Performance Summary

Our sequential **Distillation → Quantization** pipeline demonstrates the effectiveness of combining complementary optimization techniques:

#### Stage-by-Stage Analysis:
1. **Knowledge Distillation**: Compresses model architecture while preserving learned representations
2. **Dynamic Quantization**: Reduces precision for mobile deployment efficiency

#### Key Technical Insights:
- **Sequential Optimization**: Order matters - distillation first creates a smaller model that quantizes more effectively
- **Accuracy Preservation**: Knowledge distillation maintains performance through teacher-student training
- **Mobile Optimization**: INT8 quantization provides significant size reduction for deployment

#### CTO Requirements Assessment:
The pipeline successfully balances the three competing objectives:
- **Size Reduction**: Achieved through architectural compression and precision reduction
- **Speed Improvement**: Optimized for mobile hardware with INT8 operations
- **Accuracy Maintenance**: Preserved through careful knowledge transfer

This approach demonstrates that modern neural networks can be significantly optimized for mobile deployment while maintaining acceptable performance characteristics.

In [ ]:
# Cell 9: Final Analysis and Visualization
print("📊 Generating final analysis and visualizations...")

# Create comparison DataFrame
comparison_data = []

# Add baseline
comparison_data.append({
    'Model': 'Baseline',
    'Accuracy (%)': baseline_metrics['accuracy']['top1_acc'],
    'Size (MB)': baseline_metrics['size']['model_size_mb'],
    'CPU Time (ms)': baseline_metrics['timing']['cpu']['avg_time_ms'],
    'Size Reduction (%)': 0.0,
    'Accuracy Drop (%)': 0.0
})

# Add pipeline stages
baseline_size = baseline_metrics['size']['model_size_mb']
baseline_acc = baseline_metrics['accuracy']['top1_acc']

for stage_name, results in pipeline_results:
    size_reduction = (1 - results['size']['model_size_mb']/baseline_size) * 100
    acc_drop = baseline_acc - results['accuracy']['top1_acc']
    
    comparison_data.append({
        'Model': stage_name,
        'Accuracy (%)': results['accuracy']['top1_acc'],
        'Size (MB)': results['size']['model_size_mb'],
        'CPU Time (ms)': results['timing']['cpu']['avg_time_ms'],
        'Size Reduction (%)': size_reduction,
        'Accuracy Drop (%)': acc_drop
    })

# Create DataFrame and display
df_comparison = pd.DataFrame(comparison_data)
print("\n📊 COMPLETE PIPELINE COMPARISON:")
print(df_comparison.round(2))

# Save comparison results
df_comparison.to_csv('results/pipeline_comparison.csv', index=False)
print("\n💾 Results saved to results/pipeline_comparison.csv")

print("\n🎉 PIPELINE ANALYSIS COMPLETE!")